In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import torch
import torch.nn as nn
from torch import optim
from torch.utils.data import DataLoader, Dataset
import os
import pandas as pd
import numpy as np

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
df = pd.read_csv("/kaggle/input/digit-recognizer/train.csv")

In [ ]:
df.head(5)

In [ ]:
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df,train_size = 0.8, random_state = 42)
train_df = train_df.to_numpy()
test_df = test_df.to_numpy()
y_train, x_train = train_df[:,0],train_df[:,1:]
y_test, x_test = test_df[:,0],test_df[:,1:]

In [ ]:
x_train, y_train, x_test, y_test = map(torch.tensor, (x_train, y_train, x_test, y_test))

In [ ]:
x_train.shape

In [ ]:
BATCH_SIZE = 64
EPOCHS = 100
lr = 1e-2
hidden_features = 512

In [ ]:
x_train, y_train, x_test, y_test = [t.to(device) for t in [x_train, y_train, x_test, y_test]]
from torch.distributions import Uniform

In [ ]:
#INITIALISING THE WEIGHTS AND BIASES FOR TRAINING USING A NORMAL DISTRIBUTION
w1 = torch.distributions.Uniform(-np.sqrt(1/784), np.sqrt(1/784)).sample((hidden_features, 784)).to(device).requires_grad_()
b1 = torch.distributions.Uniform(-np.sqrt(1/784), np.sqrt(1/784)).sample((hidden_features, )).to(device).requires_grad_()
w2 = torch.distributions.Uniform(-np.sqrt(1/hidden_features), np.sqrt(1/hidden_features)).sample((hidden_features, hidden_features)).to(device).requires_grad_()
b2 = torch.distributions.Uniform(-np.sqrt(1/hidden_features), np.sqrt(1/hidden_features)).sample((hidden_features, )).to(device).requires_grad_()
w3 = torch.distributions.Uniform(-np.sqrt(1/hidden_features), np.sqrt(1/hidden_features)).sample((10, hidden_features)).to(device).requires_grad_()
b3 = torch.distributions.Uniform(-np.sqrt(1/hidden_features), np.sqrt(1/hidden_features)).sample((10, )).to(device).requires_grad_()

In [ ]:
def relu(z1):
    return torch.maximum(z1,torch.tensor(0., device=z1.device))

def forward(x):
    oi = relu(x@ w1.t() + b1)
    oh1 = relu(oi @ w2.t() + b2)
    oh2 = oh1 @ w3.t() + b3
    return oh2

def cross_entropy(logits, yb):
    logprobs = torch.log_softmax(logits, dim=1)
    loss = -logprobs[torch.arange(len(yb)), yb].mean()
    return loss
        




In [ ]:
print(x_train.shape)
print(w1.shape,b1.shape)
print(w2.shape,b2.shape)
print(w3.shape,b3.shape)

In [ ]:
from torch.utils.data import Dataset, DataLoader

class MNIST(Dataset) :
    def __init__(self, x, y) :
        self.x = x/255.0
        self.y = y
        self.n = len(x)

    def __len__(self) :
        return self.n

    def __getitem__(self, idx) :
        return self.x[idx], self.y[idx]
    

In [ ]:
train_ds = MNIST(x_train,y_train)
test_ds = MNIST(x_test,y_test)


In [ ]:
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle = True)
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE)

In [ ]:
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    n = 0.0
    for xb,yb in train_dl:
        logits = forward(xb)
        loss = cross_entropy(logits,yb)
        epoch_loss += loss.item()
        n += 1
        loss.backward()
        with torch.no_grad():
            w1-= w1.grad * lr
            b1-= b1.grad * lr
            w2-= w2.grad * lr
            b2-= b2.grad * lr
            w3-= w3.grad * lr
            b3-= b3.grad * lr
            w1.grad.zero_()
            b1.grad.zero_()
            w2.grad.zero_()
            b2.grad.zero_()
            w3.grad.zero_()
            b3.grad.zero_()
            
    print(epoch,epoch_loss/n)  


            

# Calculating Accuracy

In [ ]:
y_pred = forward(x_test.to(w1.dtype))
(torch.argmax(y_pred, dim=1) == y_test).to(float).mean()